## Специалист по информационным технологиям

### Агент, отвечающий на вопросы, который является специалистом по информационным технологиям
### Будет использоваться сотрудниками Insurellm, страховой технологической компании
### Агент должен быть точным, а решение должно быть недорогим.

В этом проекте будет использоваться RAG (расширенная генерация результатов поиска), чтобы обеспечить высокую точность работы нашего помощника по вопросам/ответам.

В этой первой реализации будет использоваться простой метод RAG, основанный на грубой силе..

In [1]:
# imports

import os
import glob
from pathlib import Path
import shutil
from dotenv import load_dotenv
import gradio as gr
import yaml
from tqdm import tqdm

In [2]:
# imports for langchain, plotly and Chroma

from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np
import plotly.graph_objects as go
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.embeddings import HuggingFaceEmbeddings

In [3]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4o-mini"
db_name = "vector_news_db"

In [4]:
# Load environment variables in a file called .env

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

In [5]:
# # Прочитайте документы с помощью загрузчиков LangChain
# # Найдите все из всех вложенных папок нашей базы знаний

# folders = glob.glob(r"c:/news/*")

# def add_metadata(doc, doc_type):
#     doc.metadata["doc_type"] = doc_type
#     return doc

# text_loader_kwargs = {'encoding': 'utf-8'}
# # Если это не сработает, некоторым пользователям Windows может потребоваться раскомментировать следующую строку вместо этого
# # text_loader_kwargs={'autodetect_encoding': True}

# documents = []
# for folder in folders:
#     doc_type = os.path.basename(folder)
#     loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
#     folder_docs = loader.load()
#     documents.extend([add_metadata(doc, doc_type) for doc in folder_docs])

# text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# chunks = text_splitter.split_documents(documents)

# print(f"Total number of chunks: {len(chunks)}")
# print(f"Document types found: {set(doc.metadata['doc_type'] for doc in documents)}")

In [6]:
# # Прочитайте документы с помощью загрузчиков LangChain
# # Найдите все текстовые файлы в нашей базе знаний

# # Получаем список всех файлов .md в папке news и ее подпапках
# news_files = list(Path(r"c:/news/").glob("**/*.md"))

# def add_metadata(doc, filename):
#     """Добавляет метаданные к документу."""
#     doc.metadata["date"] = os.path.splitext(filename)[0]  # Имя файла без расширения
#     if "next_bar" in doc.metadata: # Добавляем 'next_bar' только если он уже есть в метаданных
#         doc.metadata["next_bar"] = doc.metadata.get("next_bar")  # Безопасное получение, чтобы избежать KeyError
#     return doc

# text_loader_kwargs = {'encoding': 'utf-8'}
# # Если это не сработает, некоторым пользователям Windows может потребоваться раскомментировать следующую строку вместо этого
# # text_loader_kwargs={'autodetect_encoding': True}

# documents = []
# # for news_file in news_files:
# #     try:
# #         filename = os.path.basename(news_file)  # Получаем имя файла из пути
# #         loader = TextLoader(news_file, encoding=text_loader_kwargs.get('encoding')) # Используем TextLoader для чтения конкретного файла
# #         documents.extend(loader.load())  # Добавляем все документы из списка
# #         document = documents[-1]  # Получаем последний документ (единственный)
# #         document = add_metadata(document, filename)

# #     except Exception as e:
# #         print(f"Ошибка при обработке файла {news_file}: {e}")

# for news_file in news_files:
#     try:
#         filename = os.path.basename(news_file)
#         loader = TextLoader(news_file, encoding=text_loader_kwargs.get('encoding'))
#         docs = loader.load()
#         for doc in docs:
#             doc = add_metadata(doc, filename)
#             documents.append(doc)
#     except Exception as e:
#         print(f"Ошибка при обработке файла {news_file}: {e}")

# text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# chunks = text_splitter.split_documents(documents)

# print(f"Общее количество фрагментов: {len(chunks)}")
# # Выводим только те метаданные, которые у нас есть
# if documents:  # Проверяем, что документы были загружены
#     print(f"Документы даты: {set(doc.metadata['date'] for doc in documents)}")
#     print(f"Направление следующего бара: {set(doc.metadata['next_bar'] for doc in documents)}")
# else:
#     print("Не удалось найти документы.")

In [7]:
# Прочитайте документы с помощью загрузчиков LangChain
# Найдите все текстовые файлы в нашей базе знаний

# Получаем список всех файлов .md в папке news и ее подпапках
news_files = list(Path(r"c:/news/").glob("**/*.md"))

def extract_next_bar_from_md(filepath):
    with open(filepath, encoding='utf-8') as f:
        lines = f.readlines()
    if lines[0].strip() == "---":
        # Найти конец YAML-блока
        for i in range(1, len(lines)):
            if lines[i].strip() == "---":
                yaml_block = "".join(lines[1:i])
                meta = yaml.safe_load(yaml_block)
                return meta.get("next_bar")
    return None

def add_metadata(doc, file_path):
    """Добавляет метаданные к документу."""
    # filename = os.path.basename(file_path)
    # filename = file_path.stem  # Имя файла без расширения
    # doc.metadata["date"] = os.path.splitext(filename)[0]  # 
    doc.metadata["date"] = file_path.stem  # Добавление метаданных даты
    next_bar = extract_next_bar_from_md(file_path)  # Извлечение метаданных 'next_bar'
    doc.metadata["next_bar"] = next_bar  # Добавляем метаданные 'next_bar'
    return doc

text_loader_kwargs = {'encoding': 'utf-8'}
# Если это не сработает, некоторым пользователям Windows может потребоваться раскомментировать следующую строку вместо этого
# text_loader_kwargs={'autodetect_encoding': True}

documents = []

for news_file in news_files:
    try:
        # filename = os.path.basename(news_file)
        loader = TextLoader(news_file, encoding=text_loader_kwargs.get('encoding'))
        docs = loader.load()
        for doc in docs:
            doc = add_metadata(doc, news_file)
            documents.append(doc)
    except Exception as e:
        print(f"Ошибка при обработке файла {news_file}: {e}")

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Общее количество фрагментов: {len(chunks)}")
# Выводим только те метаданные, которые у нас есть
if documents:  # Проверяем, что документы были загружены
    print(f"Документы даты: {set(doc.metadata['date'] for doc in documents)}")
    print(f"Направление следующего бара: {set(doc.metadata['next_bar'] for doc in documents)}")
else:
    print("Не удалось найти документы.")

Общее количество фрагментов: 33
Документы даты: {'2025-07-07', '2025-06-26', '2025-06-27', '2025-07-08', 'current', '2025-06-30', '2025-07-03', '2025-07-09', '2025-07-14', '2025-07-16', '2025-07-04', '2025-07-10', '2025-07-15', '2025-06-25', '2025-07-01', '2025-07-11', '2025-07-02'}
Направление следующего бара: {'down', 'up', 'current'}


## Небольшое замечание о встраиваниях и "Фильмах с автоматическим кодированием"

Мы будем отображать каждый фрагмент текста в вектор, который представляет значение текста, что называется встраиванием.

Open air предлагает модель для этого, которую мы будем использовать, вызывая их API с помощью некоторого длинного кода.

Эта модель является примером "LLM с автоматическим кодированием", которая генерирует выходные данные на основе полных входных данных.
Это отличается от всех других конечностей, которые мы обсуждали сегодня, которые известны как "авторегрессивные конечности" и генерируют будущие токены только на основе прошлого контекста.

Другим примером Lms с автоматическим кодированием является BERT от Google. Помимо встраивания, Lms с автоматическим кодированием часто используются для классификации.

### Sidenote

На восьмой неделе мы вернемся к RAG и векторным встраиваниям и будем использовать векторный кодировщик с открытым исходным кодом, чтобы данные никогда не покидали наш компьютер - это важный момент при создании корпоративных систем, и данные должны оставаться внутренними.

In [8]:
# Поместите фрагменты данных в хранилище векторов, которое связывает векторное вложение с каждым фрагментом
# Chroma - популярная векторная база данных с открытым исходным кодом, основанная на SQLLite

embeddings = OpenAIEmbeddings()

# Если вы предпочитаете использовать свободные векторные вложения из HuggingFace sentence-transformers,
# то замените embeddings = OpenAIEmbeddings()
# на:
# from langchain.embeddings import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# # Удалить, если уже существует
# if os.path.exists(db_name):
#     Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()
# Удалить, если уже существует
if os.path.exists(db_name):
    try:
        shutil.rmtree(db_name)  # Удаляет всю директорию и ее содержимое
        print(f"Удалена папка {db_name}")
    except OSError as e:
        print(f"Ошибка при удалении папки {db_name}: {e}")

# # Создать vectorstore и сохранить его в папке db_name
# vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
# print(f"Векторное хранилище с {vectorstore._collection.count()} документами")

def batch(iterable, n=100):
    """Генератор для разбиения списка на батчи по n элементов"""
    l = len(iterable)
    for ndx in range(0, l, n):
        yield iterable[ndx:min(ndx + n, l)]

vectorstore = None
for chunk_batch in tqdm(list(batch(chunks, 20))):
    if vectorstore is None:
        vectorstore = Chroma.from_documents(
            documents=chunk_batch,
            embedding=embeddings,
            persist_directory=db_name
        )
    else:
        vectorstore.add_documents(chunk_batch)
print(f"Векторное хранилище с {vectorstore._collection.count()} документами")

Удалена папка vector_news_db


100%|██████████| 2/2 [00:06<00:00,  3.29s/it]

Векторное хранилище с 33 документами


In [9]:
# Давайте исследуем векторы, которые мы создали с помощью Chroma.
collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"Есть {count:,} векторов с {dimensions:,} размеры в векторном хранилище")

Есть 33 векторов с 1,536 размеры в векторном хранилище


## Визуализация хранилища векторов

Давайте на минутку взглянем на документы и векторы для их встраивания, чтобы понять, что происходит.

In [10]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['next_bar'] for metadata in metadatas]
colors = [['blue', 'red', 'black'][['up', 'down', 'current'].index(t)] for t in doc_types]

In [11]:
# Нам, людям, проще визуализировать объекты в 2D!
# Уменьшите размерность векторов до 2D, используя t-SNE
# (t-распределенное стохастическое вложение соседей)

tsne = TSNE(n_components=2, random_state=42, perplexity=5)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

# fig.update_layout(
#     title='2D Chroma Vector Store Visualization',
#     scene=dict(xaxis_title='x',yaxis_title='y'),
#     width=800,
#     height=600,
#     margin=dict(r=20, b=10, l=10, t=40)
# )

fig.update_layout(
    title='2D Chroma Vector Store Visualization',
    xaxis_title='x',
    yaxis_title='y',
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [12]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42, perplexity=5)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

## Пришло время использовать длинную цепочку, чтобы объединить все это воедино.

In [ ]:
# создайте новый чат с OpenAI
llm = ChatOpenAI(temperature=0.7, model_name=MODEL)

# Альтернатива - если вы хотите использовать Ollama локально, раскомментируйте эту строку вместо этого
# llm = ChatOpenAI(temperature=0.7, model_name='llama3.2', base_url='http://localhost:11434/v1', api_key='ollama')

# set up the conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# ретривер - это абстракция над VectorStore, которая будет использоваться во время RAG
retriever = vectorstore.as_retriever()

# сведение воедино: настройте цепочку контактов с помощью GPT 3.5 LLM, хранилища векторов и памяти
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

C:\Users\Alkor\AppData\Local\Temp\ipykernel_32096\1174663558.py:8: LangChainDeprecationWarning:

Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/



In [ ]:
# Давайте попробуем задать простой вопрос.

query = "Какие новости влияют на рост рынков за последнюю дату?"
result = conversation_chain.invoke({"question": query})
print(result["answer"])

Я не знаю, какие конкретно новости влияют на рост рынков за последнюю дату.


In [ ]:
# настройте новую память для разговоров в чате
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# собираем воедино: настройте цепочку обмена сообщениями с помощью GPT 4o-mini LLM, хранилища векторов и памяти
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

## Теперь мы расскажем об этом на радио, используя интерфейс чата -

Быстрый и простой способ создать прототип чата с LLM

In [ ]:
# Заключая это в функцию

def chat(question, history):
    result = conversation_chain.invoke({"question": question})
    return result["answer"]

In [ ]:
# Используя интерфейс Gradio:

# view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [18]:
# Давайте разберемся, что же остается за кулисы

from langchain_core.callbacks import StdOutCallbackHandler

llm = ChatOpenAI(temperature=0.7, model_name=MODEL)

memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

retriever = vectorstore.as_retriever()

conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory, callbacks=[StdOutCallbackHandler()])

query = "Какие рынки росли за последнюю дату?"
result = conversation_chain.invoke({"question": query})
answer = result["answer"]
print("\nAnswer:", answer)



> Entering new ConversationalRetrievalChain chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
- Акции Bel Fuse B достигли исторического максимума в $101,76
- TotalEnergies присоединяется к PJM Interconnection, крупнейшей электросети США
- Акции Vistra Energy Corp достигли исторического максимума в $199,84
- Глава Westinghouse Air Brake Сантана продал акции WAB на сумму $378 170
- Paysafe расширяется в Перу с новым цифровым кошельком PagoEfectivo
- MacKenzie Realty Capital одобряет обратный сплит акций 1-к-10
- S&P понизило рейтинг Ford Otomotiv до 'BB-' из-за повышенного левериджа
- Legal & General UCITS ETF объявляет о дивидендах по нескольким фондам
- Рынок акций  Италии закрылся ростом, Investing.com Италия 40 прибавил 1,62%
- FTSE 100

In [ ]:
# create a new Chat with OpenAI
llm = ChatOpenAI(temperature=0.7, model_name=MODEL)

# set up the conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# ретривер - это абстракция над VectorStore, которая будет использоваться во время RAG; k - это количество блоков, которые нужно использовать
retriever = vectorstore.as_retriever(search_kwargs={"k": 25})

# сведение воедино: настройте цепочку контактов с помощью GPT 3.5 LLM, хранилища векторов и памяти
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

In [20]:
def chat(question, history):
    result = conversation_chain.invoke({"question": question})
    return result["answer"]

In [21]:
# view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=False)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [22]:
# vectorstore.close()
# collection.close()

# Упражнения

Попробуйте применить это к своей собственной папке с данными, чтобы создать персонального специалиста по умственному развитию, эксперта по вашей собственной информации!